# Eksekusi Cuckoo Search - K-Means Clustering

Notebook ini didedikasikan secara khusus untuk **Data Preprocessing** dan **Eksekusi**. 
Seluruh tumpukan logika rumit (*back-end algoritmik*) seperti matematika Cuckoo Search, K-Means, dan perintah plot grafik visual 3D telah diekstraksi dengan sangat rapi ke file modular terpisah: `csa_core.py`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from IPython.display import display

import sys
import os
sys.path.append(os.path.abspath('../src'))

# =============== IMPORT MODULE INTI ================
from csa_core import *

# Kunci reproduktibilitas random state
np.random.seed(42)

## 1. Data Preprocessing dan Pemilihan Dimensi Fitur

In [ ]:
df_raw = pd.read_excel('../data/Data Set UMKM.xlsx', header=1)

kolom_dibutuhkan = ['Nama Usaha', 'Sektor Usaha', 'Omset per-Tahun', 'Usia', 'Laki-laki', 'Perempuan']
df = df_raw[kolom_dibutuhkan].copy()

# Membersihkan Value Kosong
df[['Usia', 'Laki-laki', 'Perempuan']] = df[['Usia', 'Laki-laki', 'Perempuan']].replace('-', 0).fillna(0)
df['Usia'] = pd.to_numeric(df['Usia'], errors='coerce').fillna(0)
df['Laki-laki'] = pd.to_numeric(df['Laki-laki'], errors='coerce').fillna(0)
df['Perempuan'] = pd.to_numeric(df['Perempuan'], errors='coerce').fillna(0)

# Persiapan Ekstraksi Dimensi Data
df['Total Pekerja Numeric'] = df['Laki-laki'] + df['Perempuan']
df['Usia Numeric'] = df['Usia']

df = df[df['Omset per-Tahun'] != '-'].dropna()
df = df.reset_index(drop=True)

def extract_omset(val):
    if 'Kurang dari 10 juta' in val:
        return 5.0
    elif '10 juta s/d 25 juta' in val:
        return 17.5
    elif '25 juta s/d 40 juta' in val:
        return 32.5
    elif '40 juta s/d 55 juta' in val:
        return 47.5
    elif '55 juta s/d 70 juta' in val:
        return 62.5
    elif '70 juta s/d 85 juta' in val:
        return 77.5
    elif '85 juta s/d 100 juta' in val:
        return 92.5
    elif '100 juta s/d 120 juta' in val:
        return 110.0
    elif '120 juta s/d 150 juta' in val:
        return 135.0
    elif 'Lebih dari 150 juta' in val:
        return 150.0
    else:
        return 5.0

df['Omset Numeric'] = df['Omset per-Tahun'].apply(extract_omset)

le = LabelEncoder()
df['Sektor Numeric'] = le.fit_transform(df['Sektor Usaha'].astype(str))

# =======================================================
# TENTUKAN KERANGKA DIMENSI DI SINI: 
# (Aktifkan baris yang tidak diawali tanda Pagar #)
# =======================================================

#(Pilihan A) KEMBALI KE 2 DIMENSI (X, Y)
# fitur_yang_dipakai = ['Sektor Numeric', 'Omset Numeric']

#(Pilihan B) MENGGUNAKAN 3 DIMENSI (X, Y, Z)
fitur_yang_dipakai = ['Sektor Numeric', 'Omset Numeric', 'Usia Numeric']

#(Pilihan C) MENGGUNAKAN 4 DIMENSI (X, Y, Z, +Warna)
# fitur_yang_dipakai = ['Sektor Numeric', 'Omset Numeric', 'Usia Numeric', 'Total Pekerja Numeric']

X_raw = df[fitur_yang_dipakai].values

# STANDARISASI
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)
print(f"Data yang digunakan diproses dalam skala {len(fitur_yang_dipakai)} Dimensi!")
display(df[['Nama Usaha'] + fitur_yang_dipakai].head(5))

## 2. Penentuan Optimal Kluster otomatis (Elbow L-Bow Method)

In [ ]:
# Memanggil fungsi ajaib dari csa_core.py
print("Mengkalkulasi K-Optimal...")
optimal_k = hitung_optimal_k_elbow(X_scaled, max_k=10)
print(f"Nilai Evaluasi Algoritma Elbow menyarankan Kluster terbaik adalah K : {optimal_k}")

## 3. Eksekusi Program Model (Cuckoo Search → K-Means)

In [ ]:
print(f"Mengeksekusi Iterasi Cuckoo Search Algoritma Pada Data {len(fitur_yang_dipakai)} Dimensi...")
best_cuckoo_centroids_scaled, iterations_log = cuckoo_search_kmeans(X_scaled, k=optimal_k, n_nests=10, max_iter=30, pa=0.25)

# Memasukkan Centroid Optimal Ke Algoritma Dasar Final
final_labels, kmeans_centroids_scaled = final_kmeans(X_scaled, best_cuckoo_centroids_scaled)

df['Cluster'] = final_labels
df['Cluster'] = df['Cluster'].apply(lambda x: f"C{int(x)+1}")

print("\n================ TABEL HASIL AKHIR CLUSTERING ================")
display(df[['Nama Usaha'] + fitur_yang_dipakai + ['Cluster']].head(10))

## 4. Visualisasi Sebaran Fitur

In [ ]:
# Grafik secara pintar akan melihat jumlah dimensi dan melukis plot Sumbu X Y Z yang bisa di rotasi!
plot_hasil_cluster(X_scaled, kmeans_centroids_scaled, final_labels, fitur_yang_dipakai)